<a href="https://colab.research.google.com/github/czechuuu/micro-vla/blob/main/MicroVla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MIMUW Reinforcement Learning 26/26L

Adrian Boguszewski, Bartosz Czechowski

# Project statement
## Overview & Motivation
Vision-Language-Action (VLA) models, such as RT-2 and OpenVLA, have recently demonstrated incredible zero-shot generalization in robotics. However, these models face a severe data bottleneck: while we have massive repositories of video data showing humans or robots completing tasks, we have very little action-annotated data containing the exact motor torques and joint commands required to replicate those movements.
Standard supervised approaches fail when actions are missing. If we can design architectures that learn world dynamics and physics directly from "observation-only" videos, we can vastly expand the training sets for robotic foundation models. This project explores whether pre-training on a large corpus of action-free manipulation videos can improve a small-scale VLA's ability to perform In-Context Learning (ICL) from just one or two fully annotated demonstrations.
## Project Objective
The primary goal of this project is to build a "Micro-VLA" agent that improves its sample efficiency and few-shot generalization by learning from unannotated video trajectories. The student will implement a system that uses an Inverse Dynamics Model (or masked token prediction) to ingest observation-only sequences from a simulated robotics environment, combining this with a small set of action-annotated data to solve continuous control manipulation tasks.
## Expected Deliverables (Pass Criteria)
To successfully pass this project, students are expected to complete the following concrete tasks:
- **Simulated Benchmark Setup**: Set up a lightweight, continuous-control robotic manipulation environment (e.g., Robomimic or Meta-World). Generate a synthetic dataset: a small fraction containing full state-action-reward data, and a large fraction stripped of actions to simulate "video-only" observations.
- **Architecture Implementation**: Design a small Transformer-based policy (e.g., a miniaturized Decision Transformer). Implement an auxiliary objective, such as an Inverse Dynamics module, that forces the network to predict the missing actions between two consecutive video frames, allowing it to learn from the observation-only dataset.
- **Benchmarking & Evaluation**: Train two models: a baseline Micro-VLA trained strictly on the small, fully-annotated dataset, and your proposed Micro-VLA trained on the mixed dataset.
- **Comparative Analysis**: Deliver a final report evaluating the models on their In-Context Learning capabilities. Specifically, prompt the frozen models with a single successful demonstration of a new, unseen task variation (e.g., picking up a differently colored object) and measure which model exhibits better zero-shot or few-shot transfer.


# 0. Configs

In [65]:
import os
import numpy as np
import robosuite as suite
import torch

In [115]:
class Config:
    # --- Environment Settings ---
    env_name = "pick-place-v3"
    max_steps = 150

    # --- Observation Settings ---
    image_size = (200, 200)
    cameras = ["corner2", "topview"]

    # --- Dataset Generation Settings ---
    num_annotated_episodes = 1
    num_video_episodes = 1
    save_dir = "./micro_vla_data"

    # --- Model/Training Settings ---
    device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Simulated Benchmark Setup


## Prerequisites

In [6]:
import os

# Tell MuJoCo to use a headless, off-screen rendering backend.
# 'egl' is best if you have a GPU (like on Colab).
# 'osmesa' or 'glfw' are fallbacks if EGL isn't installed.
os.environ['MUJOCO_GL'] = 'egl'

In [55]:
! pip install metaworld

## Functions for generating data

In [131]:
import os
import random
import numpy as np
import metaworld
import mujoco
from metaworld.policies import * # Imports all expert policies

def get_policy_for_env(env_name):
    """
    Dynamically finds the expert policy class for a given environment name.
    Example: 'pick-place-v2' -> SawyerPickPlaceV2Policy
    """
    # 1. Format name: 'pick-place-v2' -> 'PickPlaceV2'
    formatted_name = "".join([word.capitalize() for word in env_name.split('-')])
    # 2. Add prefix/suffix: 'SawyerPickPlaceV2Policy'
    policy_class_name = f"Sawyer{formatted_name}Policy"

    # 3. Fetch from metaworld.policies
    try:
        policy_class = globals().get(policy_class_name)
        if policy_class is None:
            # Fallback for some naming variations in older/newer versions
            import metaworld.policies as policies
            policy_class = getattr(policies, policy_class_name)
        return policy_class()
    except AttributeError:
        raise ValueError(f"Could not find expert policy for {env_name} (looked for {policy_class_name})")

def setup_environment(config):
    """Initializes the environment, policy, and renderer."""
    mt1 = metaworld.MT1(config.env_name)
    env = mt1.train_classes[config.env_name]()

    # Set a random task variation
    task = random.choice(mt1.train_tasks)
    env.set_task(task)

    # Initialize the expert policy
    policy = get_policy_for_env(config.env_name)

    # Initialize the native MuJoCo renderer
    renderer = mujoco.Renderer(
        env.unwrapped.model,
        height=config.image_size[0],
        width=config.image_size[1]
    )
    return env, policy, renderer

def get_action(policy, obs):
    """Uses the expert policy to generate an action based on the current state."""
    return policy.get_action(obs)

def collect_episode(env, policy, renderer, config):
    """Runs one full episode using the expert policy and collects all data."""
    # Handle Gymnasium version differences for reset safely
    reset_result = env.reset()
    obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result

    episode_data = {
        "states": [],
        "actions": []
    }
    for cam in config.cameras:
        episode_data[f"images_{cam}"] = []

    success_achieved = False

    for step in range(config.max_steps):
        # 1. Render all camera views
        for cam in config.cameras:
            renderer.update_scene(env.unwrapped.data, camera=cam)
            img = renderer.render()
            episode_data[f"images_{cam}"].append(img)

        # 2. Record state and expert action
        action = get_action(policy, obs)
        episode_data["states"].append(obs)
        episode_data["actions"].append(action)

        # 3. Step the environment
        step_result = env.step(action)
        # Handle Gym 0.21 vs 0.26 API differences safely
        if len(step_result) == 5:
            obs, reward, done, truncated, info = step_result
        else:
            obs, reward, done, info = step_result
            truncated = False

        # --- DEBUGGING LOGIC ---
        # info['success'] turns to 1.0 when the object is within the 3D target radius
        if info.get('success', 0) == 1.0 and not success_achieved:
            print(f"    [Debug] Task SUCCESS! Robot reached the 3D goal at step {step}.")
            success_achieved = True

        if done or truncated:
            break

    # Convert to numpy arrays
    for key in episode_data:
        dtype = np.float32 if key in ["states", "actions"] else np.uint8
        episode_data[key] = np.array(episode_data[key], dtype=dtype)

    # If the episode finished but it never hit the success state, print why
    if not success_achieved:
        dist = info.get('obj_to_target', 'Unknown')
        print(f"    [Debug] Episode ended WITHOUT success. Final distance to 3D goal: {dist:.4f}")

    return episode_data

def save_dataset(episodes, folder_path, dataset_name, keep_actions=True):
    """Saves a list of episode dictionaries to disk using NumPy's compressed format."""
    os.makedirs(folder_path, exist_ok=True)

    # If it's the observation-only dataset, delete the actions!
    if not keep_actions:
        for ep in episodes:
            if "actions" in ep:
                del ep["actions"]

    file_path = os.path.join(folder_path, f"{dataset_name}.npz")

    # np.savez_compressed expects keyword arguments, so we unpack a dictionary
    np.savez_compressed(file_path, episodes=episodes)
    print(f"Saved {len(episodes)} episodes to {file_path} | Actions included: {keep_actions}")

In [132]:
import random

def generate_all_data(config):
    print("🚀 Initializing Meta-World environment and Expert Policy...")
    # Unpack the three objects from our updated setup function
    env, policy, renderer = setup_environment(config)

    # We will use the MT1 task list to vary the environment for every episode
    mt1 = metaworld.MT1(config.env_name)
    all_tasks = mt1.train_tasks

    # --- 1. Generate Small Annotated Dataset ---
    print(f"\n--- Phase 1: Generating {config.num_annotated_episodes} Annotated Episodes ---")
    annotated_episodes = []
    for i in range(config.num_annotated_episodes):
        # Set a new random task variation for each episode
        env.set_task(random.choice(all_tasks))

        # Pass the policy into the collection step
        ep_data = collect_episode(env, policy, renderer, config)
        annotated_episodes.append(ep_data)

        if (i+1) % 2 == 0 or i == 0:
            print(f"✅ Collected annotated episode {i+1}/{config.num_annotated_episodes}")

    save_dataset(annotated_episodes, config.save_dir, "dataset_annotated", keep_actions=True)

    # --- 2. Generate Large Observation-Only Dataset ---
    print(f"\n--- Phase 2: Generating {config.num_video_episodes} Video-Only Episodes ---")
    video_episodes = []
    for i in range(config.num_video_episodes):
        # Set a new random task variation for each episode
        env.set_task(random.choice(all_tasks))

        # Even for video-only data, we use the expert policy so the model
        # learns from successful physics/dynamics.
        ep_data = collect_episode(env, policy, renderer, config)
        video_episodes.append(ep_data)

        if (i+1) % 10 == 0:
            print(f"🎬 Collected video episode {i+1}/{config.num_video_episodes}")

    save_dataset(video_episodes, config.save_dir, "dataset_videos", keep_actions=False)

    print("\n✨ All datasets generated successfully")

## Generating the data

In [133]:
generate_all_data(Config)

🚀 Initializing Meta-World environment and Expert Policy...

--- Phase 1: Generating 1 Annotated Episodes ---


/usr/local/lib/python3.12/dist-packages/metaworld/policies/policy.py:49: UserWarning: Constant(s) may be too high. Environments clip response to [-1, 1]
  warnings.warn(


    [Debug] Task SUCCESS! Robot reached the 3D goal at step 53.
✅ Collected annotated episode 1/1
Saved 1 episodes to ./micro_vla_data/dataset_annotated.npz | Actions included: True

--- Phase 2: Generating 1 Video-Only Episodes ---
    [Debug] Task SUCCESS! Robot reached the 3D goal at step 51.
Saved 1 episodes to ./micro_vla_data/dataset_videos.npz | Actions included: False

✨ All datasets generated successfully


## Visualising the trajetories

In [88]:
import numpy as np
import imageio
import os
import cv2
from IPython.display import HTML, display
import base64

def create_interactive_trajectory(file_path, episode_idx=0, camera_name="agentview", output_name="trajectory.mp4", fps=20, scale=4.0, flip_upside_down=True):
    """
    Creates a loopable MP4 video with a seek bar for a specific trajectory.
    """
    if not os.path.exists(file_path):
        print(f"❌ Error: File {file_path} not found.")
        return

    data = np.load(file_path, allow_pickle=True)
    episodes = data['episodes']

    img_key = f"images_{camera_name}"
    raw_frames = episodes[episode_idx][img_key]
    processed_frames = []

    # 1. Process Frames (Flip and Resize)
    for frame in raw_frames:
        if flip_upside_down:
            frame = np.flip(frame, axis=0)

        if scale != 1.0:
            h, w, c = frame.shape
            new_dims = (int(w * scale), int(h * scale))
            frame = cv2.resize(frame, new_dims, interpolation=cv2.INTER_LINEAR)

        processed_frames.append(frame)

    # 2. Save as MP4
    # imageio uses ffmpeg under the hood for mp4
    imageio.mimsave(output_name, processed_frames, fps=fps, macro_block_size=1)
    print(f"✅ Video saved to: {output_name}")

    # 3. Display with HTML5 Controls (Status bar + Skip + Loop)
    video_file = open(output_name, "rb").read()
    video_url = f"data:video/mp4;base64,{base64.b64encode(video_file).decode()}"

    # This HTML snippet adds the seek bar ('controls') and 'loop'
    html_code = f"""
    <video width="{processed_frames[0].shape[1]}" controls loop autoplay muted>
        <source src="{video_url}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    """
    display(HTML(html_code))

In [135]:
create_interactive_trajectory("./micro_vla_data/dataset_annotated.npz", episode_idx=0, camera_name="corner2")

✅ Video saved to: trajectory.mp4


# 2. Architecture implementation